# Data Cleaning — Pipeline Tag Prediction
**DATASCI 266: Natural Language Processing with Deep Learning**
UC Berkeley, School of Information

---

This notebook cleans the Hugging Face model-cards dataset **before** it's fed into
`pipeline_tag_prediction.ipynb`. It runs in this order:

1. Load raw dataset
2. Handle missing values
3. Check for label leakage
4. Strip YAML frontmatter from card text
5. Deduplicate (exact + near-duplicate/boilerplate)
6. Quality / length filtering
7. Top-N tag selection & class balance decision
8. Final summary & sanity checks
9. Save the cleaned dataset so it can be loaded from a different notebook

Dedup and splitting order matters: **dedup happens before any train/val/test split**
(done in the training notebook), so near-identical cards can't leak across splits.


## 0. Setup

In [ ]:
# Install dependencies
!pip install -q datasets pandas pyarrow huggingface_hub

In [ ]:
import os
import re
import json
import hashlib
import random

import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_colwidth', 120)

## 1. Load Raw Dataset

Same source as the training notebook: `librarian-bots/model_cards_with_metadata`.
Note this is a curated crawl (~515K rows as of mid-2026), not a full mirror of every
model on the Hub (2M+) — see the discussion in the training notebook for why that's
an acceptable tradeoff for this project.

In [ ]:
print("Loading dataset...")
ds = load_dataset("librarian-bots/model_cards_with_metadata", split="train") #only has one split: "train"
df_raw = ds.to_pandas()

TEXT_COL = 'card'  # <-- confirm this matches the actual schema; see training notebook note
assert TEXT_COL in df_raw.columns, f"Column '{TEXT_COL}' not found. Available: {df_raw.columns.tolist()}"

print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)

Loading dataset...
Raw shape: (667424, 10)


,modelId,author,last_modified,downloads,likes,library_name,tags,pipeline_tag,createdAt,card
0,batiai/Qwen3.5-35B-A3B-GGUF,batiai,2026-05-04 04:49:09+00:00,1019,0,llama.cpp,"[llama.cpp, gguf, qwen, moe, quantized, apple-silicon, ollama, batiai, on-device, text-generation, en, ko, ja, zh, b...",text-generation,2026-04-13 13:37:21+00:00,---\nlanguage:\n - en\n - ko\n - ja\n - zh\nlicense: apache-2.0\ntags:\n - gguf\n - qwen\n - moe\n - quantiz...
1,johneze/ktt-math-tutor-lora,johneze,2026-04-24 12:35:08+00:00,0,0,peft,"[peft, safetensors, base_model:adapter:HuggingFaceTB/SmolLM2-135M-Instruct, lora, transformers, text-generation, con...",text-generation,2026-04-24 12:35:04+00:00,---\nbase_model: HuggingFaceTB/SmolLM2-135M-Instruct\nlibrary_name: peft\npipeline_tag: text-generation\ntags:\n- ba...
2,elVlad/bge-m3-Q4_K_M-GGUF,elVlad,2026-07-07 19:20:52+00:00,0,1,sentence-transformers,"[sentence-transformers, gguf, feature-extraction, sentence-similarity, llama-cpp, gguf-my-repo, base_model:BAAI/bge-...",sentence-similarity,2026-07-07 19:20:45+00:00,---\npipeline_tag: sentence-similarity\ntags:\n- sentence-transformers\n- feature-extraction\n- sentence-similarity\...


In [ ]:
# Track row counts at every stage so the cleaning report is auditable
stage_counts = [("raw", len(df_raw))]
df = df_raw.copy()

## 2. Handle Missing Values

Drop rows with no `pipeline_tag` (can't be a training label) and rows with
empty/null card text (nothing to learn from). Log the drop counts explicitly —
the raw fraction with no `pipeline_tag` set is worth reporting as a known
limitation of Hub metadata.

In [ ]:
n_before = len(df)
n_missing_tag = df['pipeline_tag'].isna().sum()
n_missing_text = df[TEXT_COL].isna().sum() | (df[TEXT_COL].fillna('').str.strip() == '')

print(f"Rows missing pipeline_tag: {n_missing_tag:,} ({100*n_missing_tag/n_before:.1f}%)")

df = df[df['pipeline_tag'].notna()].copy()
df = df[df[TEXT_COL].notna() & (df[TEXT_COL].str.strip() != '')].copy()

print(f"Rows dropped (missing tag or text): {n_before - len(df):,}")
print(f"Remaining: {len(df):,}")

stage_counts.append(("after missing-value drop", len(df)))

Rows missing pipeline_tag: 329,733 (49.4%)
Rows dropped (missing tag or text): 329,733
Remaining: 337,691


## 3. Check for Label Leakage

`pipeline_tag` is model card metadata, and it commonly appears verbatim inside
the YAML frontmatter at the top of the card — which is part of the raw `text`
column. If we train on that raw text, the model could just be pattern-matching
the string `pipeline_tag: <tag>` back out of the input, which would make
reported accuracy meaningless. We check this **before** stripping YAML, so we
know exactly how big the problem is.

In [ ]:
def contains_tag_string(row):
    txt = row[TEXT_COL]
    tag = row['pipeline_tag']
    if not isinstance(txt, str) or not isinstance(tag, str):
        return False
    return ('pipeline_tag' in txt) or (tag.lower() in txt.lower())

leak_mask = df.apply(contains_tag_string, axis=1)
n_leak = leak_mask.sum()

print(f"Rows where raw text contains 'pipeline_tag' or the tag string: {n_leak:,} "
      f"({100*n_leak/len(df):.1f}%)")
print("This is expected — the YAML frontmatter block includes pipeline_tag directly.")
print("Resolved in the next step by stripping YAML frontmatter before tokenization.")

Rows where raw text contains 'pipeline_tag' or the tag string: 250,503 (74.2%)
This is expected — the YAML frontmatter block includes pipeline_tag directly.
Resolved in the next step by stripping YAML frontmatter before tokenization.


## 4. Strip YAML Frontmatter

Model cards on the Hub start with a `---\n ... \n---` YAML metadata block
(license, tags, pipeline_tag, base_model, etc.) followed by the actual README
prose. We strip that block out and train only on the prose — this directly
fixes the leakage identified above, and also means the model is learning from
actual descriptive content rather than metadata.

We keep the stripped YAML separately (not used for training labels, just kept
for reference/debugging) rather than discarding it silently.

In [ ]:
YAML_BLOCK_RE = re.compile(r'^---\s*\n.*?\n---\s*\n?', re.DOTALL)

def strip_yaml_frontmatter(text):
    if not isinstance(text, str):
        return text, None
    match = YAML_BLOCK_RE.match(text)
    if match:
        yaml_block = match.group(0)
        body = text[match.end():]
        return body.strip(), yaml_block
    return text.strip(), None

stripped = df[TEXT_COL].apply(strip_yaml_frontmatter)
df['text_clean'] = stripped.apply(lambda x: x[0])
df['yaml_block'] = stripped.apply(lambda x: x[1])

n_had_yaml = df['yaml_block'].notna().sum()
print(f"Rows with a detected YAML frontmatter block: {n_had_yaml:,} ({100*n_had_yaml/len(df):.1f}%)")

# Re-check leakage on the cleaned text column
leak_after = df.apply(lambda r: contains_tag_string({**r, TEXT_COL: r['text_clean']}), axis=1)
print(f"Rows where CLEANED text still contains tag string: {leak_after.sum():,}")

Rows with a detected YAML frontmatter block: 334,034 (98.9%)
Rows where CLEANED text still contains tag string: 47,492


## 5. Deduplicate Cards

Two duplication patterns are common in Hub crawls:

- **Exact duplicates** — identical card body across different `modelId`s
  (GGUF requantizations, LoRA variant reposts, etc.)
- **Boilerplate / template duplicates** — cards that are 95%+ auto-generated
  scaffold text (e.g. the default `transformers` push_to_hub template) with
  only the model name swapped in. These carry almost no signal for a tag
  classifier and can cause the model to overfit to boilerplate phrasing.

We catch both with a normalized-text hash: lowercase, collapse whitespace,
strip digits and the model's own name/id token before hashing, so that two
cards differing only in a model name or run number still hash identically.

In [ ]:
def normalize_for_hash(text, model_id=None):
    if not isinstance(text, str):
        return ''
    t = text.lower()
    if isinstance(model_id, str):
        # remove the model's own name/id so it doesn't make an otherwise-identical
        # boilerplate card look unique
        name_part = model_id.split('/')[-1].lower()
        t = t.replace(model_id.lower(), '')
        t = t.replace(name_part, '')
    t = re.sub(r'\d+', '', t)                # strip numbers (run ids, timestamps, versions)
    t = re.sub(r'[^a-z\s]', ' ', t)           # strip punctuation
    t = re.sub(r'\s+', ' ', t).strip()        # collapse whitespace
    return t

df['norm_hash'] = df.apply(
    lambda r: hashlib.md5(normalize_for_hash(r['text_clean'], r.get('modelId')).encode('utf-8')).hexdigest(),
    axis=1
)

n_before_dedup = len(df)
dup_counts = df['norm_hash'].value_counts()
n_dup_groups = (dup_counts > 1).sum()
n_dup_rows = dup_counts[dup_counts > 1].sum()

print(f"Duplicate groups found: {n_dup_groups:,}")
print(f"Rows involved in duplication: {n_dup_rows:,} ({100*n_dup_rows/n_before_dedup:.1f}%)")

# Keep exactly one representative per duplicate group (first by modelId for determinism)
df = (df.sort_values('modelId')
        .drop_duplicates(subset='norm_hash', keep='first')
        .reset_index(drop=True))

print(f"Rows after dedup: {len(df):,} (dropped {n_before_dedup - len(df):,})")
stage_counts.append(("after dedup", len(df)))

Duplicate groups found: 17,160
Rows involved in duplication: 189,050 (56.0%)
Rows after dedup: 165,801 (dropped 171,890)


In [ ]:
# Inspect a few of the largest duplicate groups to sanity-check the approach
print("Largest duplicate groups (by normalized text hash):")
print(dup_counts[dup_counts > 1].head(10))

Largest duplicate groups (by normalized text hash):
norm_hash
ee80aefe6c43e61a227ef318a75b23f3    47238
80decf66246b936e4f70af783b8cae21    17652
9c48e045ef5688e3acb09915e0499344     7237
1da4a1d91451b1270bd20aa5716364da     6073
22a4e77ff2bc41ed0ce3db08643d40da     4908
d41d8cd98f00b204e9800998ecf8427e     3993
fc8e9fdab9cecffa3ee3e652e7edefcd     2275
82bf52ab3f971847f9f02b691375005d     1807
cb42579b32048d65223b408a627138f8     1329
f137a2e99fe01a3a5a2c588cc6ec4536     1147
Name: count, dtype: int64


## 6. Quality / Length Filtering

Beyond a raw character-count minimum, we look at information density: a card
padded with YAML/license boilerplate but almost no unique prose isn't useful
even if it clears a length threshold. We use unique-word ratio as a cheap
proxy after YAML stripping.

In [ ]:
MIN_CHAR_LEN = 100
MAX_CHAR_LEN = 100_000
MIN_UNIQUE_WORD_RATIO = 0.15   # unique words / total words in the cleaned body

df['char_len'] = df['text_clean'].fillna('').apply(len)
df['word_count'] = df['text_clean'].fillna('').apply(lambda x: len(x.split()))
df['unique_word_ratio'] = df['text_clean'].fillna('').apply(
    lambda x: (len(set(x.lower().split())) / len(x.split())) if len(x.split()) > 0 else 0
)

n_before_quality = len(df)

df = df[
    (df['char_len'] >= MIN_CHAR_LEN) &
    (df['char_len'] <= MAX_CHAR_LEN) &
    (df['unique_word_ratio'] >= MIN_UNIQUE_WORD_RATIO)
].copy()

print(f"Rows dropped by length/quality filter: {n_before_quality - len(df):,}")
print(f"Remaining: {len(df):,}")
stage_counts.append(("after quality filter", len(df)))

Rows dropped by length/quality filter: 914
Remaining: 164,887


## 7. Top-N Tag Selection & Class Balance

Select the top N most frequent tags (same as the training notebook), then
decide explicitly how to handle imbalance rather than leaving it implicit.
`CAP_MAJORITY_CLASSES` is a toggle — set `None` to leave distribution as-is
(and rely on macro-F1 / class weighting at training time), or set an integer
to cap any class at that many rows via random undersampling.

In [ ]:
TOP_N_TAGS = 10
CAP_MAJORITY_CLASSES = None  # e.g. 20_000 to cap; None to leave imbalance as-is

top_tags = df['pipeline_tag'].value_counts().head(TOP_N_TAGS).index.tolist()
print(f"Selected tags: {top_tags}")

df = df[df['pipeline_tag'].isin(top_tags)].copy()
print(f"Rows after top-{TOP_N_TAGS} tag filter: {len(df):,}")
print(df['pipeline_tag'].value_counts())

if CAP_MAJORITY_CLASSES is not None:
    df = (df.groupby('pipeline_tag', group_keys=False)
            .apply(lambda g: g.sample(min(len(g), CAP_MAJORITY_CLASSES), random_state=SEED))
            .reset_index(drop=True))
    print(f"\nAfter capping majority classes at {CAP_MAJORITY_CLASSES:,}:")
    print(df['pipeline_tag'].value_counts())

stage_counts.append(("after top-N tag filter / balancing", len(df)))

Selected tags: ['text-generation', 'text-to-image', 'image-text-to-text', 'text-classification', 'translation', 'automatic-speech-recognition', 'token-classification', 'robotics', 'sentence-similarity', 'image-classification']
Rows after top-10 tag filter: 139,275
pipeline_tag
text-generation                 66536
text-to-image                   21632
image-text-to-text              12035
text-classification             10095
translation                      8551
automatic-speech-recognition     5705
token-classification             3937
robotics                         3818
sentence-similarity              3697
image-classification             3269
Name: count, dtype: int64


In [ ]:
# Encode labels (kept consistent with the training notebook's convention)
label2id = {tag: i for i, tag in enumerate(sorted(top_tags))}
id2label = {i: tag for tag, i in label2id.items()}
df['label'] = df['pipeline_tag'].map(label2id)
print(f"Label mapping: {label2id}")

Label mapping: {'automatic-speech-recognition': 0, 'image-classification': 1, 'image-text-to-text': 2, 'robotics': 3, 'sentence-similarity': 4, 'text-classification': 5, 'text-generation': 6, 'text-to-image': 7, 'token-classification': 8, 'translation': 9}


## 8. Final Summary & Sanity Checks

In [ ]:
print("=== Row counts by cleaning stage ===")
for stage, count in stage_counts:
    print(f"{stage:35s}: {count:>10,}")

print("\n=== Final class distribution ===")
print(df['pipeline_tag'].value_counts())

print("\n=== Final columns kept ===")
KEEP_COLS = ['modelId', 'pipeline_tag', 'label', 'text_clean', 'char_len', 'word_count']
KEEP_COLS = [c for c in KEEP_COLS if c in df.columns]
print(KEEP_COLS)

df_final = df[KEEP_COLS].rename(columns={'text_clean': 'text'}).reset_index(drop=True)
df_final.head(3)

=== Row counts by cleaning stage ===
raw                                :    667,424
after missing-value drop           :    337,691
after dedup                        :    165,801
after quality filter               :    164,887
after top-N tag filter / balancing :    139,275

=== Final class distribution ===
pipeline_tag
text-generation                 66536
text-to-image                   21632
image-text-to-text              12035
text-classification             10095
translation                      8551
automatic-speech-recognition     5705
token-classification             3937
robotics                         3818
sentence-similarity              3697
image-classification             3269
Name: count, dtype: int64

=== Final columns kept ===
['modelId', 'pipeline_tag', 'label', 'text_clean', 'char_len', 'word_count']


,modelId,pipeline_tag,label,text,char_len,word_count
0,00000tt/OmniDream-LoRas,text-to-image,7,# OmniDream FinalDream LoRas\n\n<Gallery />\n\n## Model description \n\nOmniDream FinalDream LoRA weights for Wan2.2...,305,40
1,00001a/gemma-4-26B-A4B-it-uncensored-heretic-ara-MLX-6bit-int6-affine,image-text-to-text,2,# 🦆 zecanard/gemma-4-26B-A4B-it-uncensored-heretic-ara-MLX-6bit-int6-affine\n\n[This model](https://huggingface.co/z...,1343,139
2,001szp/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled,image-text-to-text,2,# 🌟 Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\n\n> **Build Environment Upgrades:**\n> - **Fine-tuning Framewor...,7758,919


## 9. Save the Cleaned Dataset

Two things get saved:

1. **The cleaned data** as Parquet (compact, preserves dtypes, fast to reload —
   better than CSV for this size and avoids re-quoting large text fields).
2. **A metadata sidecar** (`label2id`, `id2label`, the tag list, and the
   cleaning stage counts) as JSON, so a different notebook doesn't have to
   re-derive the label mapping and risk it coming out in a different order.

### Making it accessible from a different notebook
Pick **one** of the two options below depending on your setup:

- **Google Drive** (simplest if both notebooks run in Colab under your account) —
  mount Drive and save there; the second notebook mounts Drive and reads the same path.
- **Hugging Face Hub** (better if you want it accessible outside Colab too, or
  want versioning) — push as a private dataset repo; the second notebook just
  calls `load_dataset(...)` with your repo id.

Both cells below are written to run as-is; the Drive cell will prompt an auth
flow the first time, and the Hub cell needs a real repo id + a token with
write access (via `huggingface-cli login` or `notebook_login()`).

In [ ]:
OUTPUT_DIR = './cleaned_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

parquet_path = os.path.join(OUTPUT_DIR, 'model_cards_cleaned.parquet')
metadata_path = os.path.join(OUTPUT_DIR, 'metadata.json')

df_final.to_parquet(parquet_path, index=False)

metadata = {
    'label2id': label2id,
    'id2label': id2label,
    'top_tags': top_tags,
    'text_col': 'text',
    'stage_counts': stage_counts,
    'min_char_len': MIN_CHAR_LEN,
    'max_char_len': MAX_CHAR_LEN,
    'min_unique_word_ratio': MIN_UNIQUE_WORD_RATIO,
    'seed': SEED,
}
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved cleaned data:  {parquet_path}  ({os.path.getsize(parquet_path)/1e6:.1f} MB)")
print(f"Saved metadata:      {metadata_path}")

Saved cleaned data:  ./cleaned_data/model_cards_cleaned.parquet  (217.7 MB)
Saved metadata:      ./cleaned_data/metadata.json


### Option A — Google Drive (uncomment to use)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_DIR = '/content/drive/MyDrive/266-pipeline-tag-prediction'
# os.makedirs(DRIVE_DIR, exist_ok=True)
#
# import shutil
# shutil.copy(parquet_path, os.path.join(DRIVE_DIR, 'model_cards_cleaned.parquet'))
# shutil.copy(metadata_path, os.path.join(DRIVE_DIR, 'metadata.json'))
# print(f"Copied to Drive: {DRIVE_DIR}")

### Option B — Hugging Face Hub (uncomment to use)

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()  # paste a token with write access
#
# from datasets import Dataset
#
# HF_REPO_ID = "your-username/pipeline-tag-cleaned"  # <-- set this
#
# hf_ds = Dataset.from_pandas(df_final)
# hf_ds.push_to_hub(HF_REPO_ID, private=True)
#
# # Push the metadata sidecar alongside it
# from huggingface_hub import HfApi
# HfApi().upload_file(
#     path_or_fileobj=metadata_path,
#     path_in_repo="metadata.json",
#     repo_id=HF_REPO_ID,
#     repo_type="dataset",
# )
# print(f"Pushed to https://huggingface.co/datasets/{HF_REPO_ID}")

### Loading from a different notebook

```python
# --- If saved to Google Drive ---
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, json

DRIVE_DIR = '/content/drive/MyDrive/266-pipeline-tag-prediction'
df = pd.read_parquet(f'{DRIVE_DIR}/model_cards_cleaned.parquet')
metadata = json.load(open(f'{DRIVE_DIR}/metadata.json'))
label2id, id2label = metadata['label2id'], metadata['id2label']

# --- If pushed to Hugging Face Hub ---
from datasets import load_dataset
ds = load_dataset("your-username/pipeline-tag-cleaned", split="train")
df = ds.to_pandas()
```
